# 🕸️ Day 21 — Graph Neural Networks, Meta-Learning & Continual Learning
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | Graph Neural Networks: Message Passing | ~11 sec |
| 2 | 10:30–11:00 AM | Prototypical Networks (Few-Shot Learning) | ~5 sec |
| 3 | 11:00 AM–1:00 PM | MAML + Continual Learning (EWC + Replay) | ~18 sec |
| 4 | 1:00–2:00 PM | Lunch Break | — |
| 5 | 2:00–4:00 PM | Research Capstone: GNN + Meta + Continual | ~20 sec |

> **Zero downloads. Pure numpy + sklearn + scipy.**
---

In [ ]:
# !pip install scikit-learn networkx matplotlib pandas numpy scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import networkx as nx
import warnings, time
warnings.filterwarnings('ignore')

from sklearn.datasets import load_wine, load_iris, load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import accuracy_score
import sklearn
print('All imports OK')
print(f'   sklearn   : {sklearn.__version__}')
print(f'   networkx  : {nx.__version__}')

---
## Session 1 — Graph Neural Networks: Message Passing
**Time:** 9:00–10:30 AM | **Run time: ~11 sec**

In [ ]:
# 1.1  GCN Layer: Spectral Graph Convolution
def gcn_layer(H, A, W, activation=np.tanh):
    """
    GCN layer: H_new = activation(A_norm @ H @ W)
    where A_norm = D^(-1/2) (A + I) D^(-1/2)  — symmetric normalisation
    """
    N     = len(A)
    A_hat = A + np.eye(N)                         # add self-loops
    D_vec = A_hat.sum(axis=1)                      # degree vector
    D_inv_sqrt = np.diag(D_vec ** -0.5)
    A_norm     = D_inv_sqrt @ A_hat @ D_inv_sqrt   # symmetric normalisation
    return activation(A_norm @ H @ W)

# Build a small social network graph
rng = np.random.default_rng(42)
N   = 8    # nodes
D_IN, D_H1, D_H2 = 5, 8, 4   # feature dims

# Adjacency matrix (undirected)
A = np.array([
    [0,1,1,0,0,0,0,0],
    [1,0,1,1,0,0,0,0],
    [1,1,0,1,0,0,0,0],
    [0,1,1,0,1,0,0,0],
    [0,0,0,1,0,1,1,0],
    [0,0,0,0,1,0,1,1],
    [0,0,0,0,1,1,0,1],
    [0,0,0,0,0,1,1,0],
], dtype=float)

# Node features
H0 = rng.normal(0, 1, (N, D_IN))
# GCN weights
W1 = rng.normal(0, 0.1, (D_IN, D_H1))
W2 = rng.normal(0, 0.1, (D_H1, D_H2))

# Two GCN layers
H1 = gcn_layer(H0, A, W1)
H2 = gcn_layer(H1, A, W2)

print('GCN Forward Pass:')
print(f'  Input node features  : {H0.shape}  (nodes × features)')
print(f'  After GCN layer 1    : {H1.shape}  (nodes × {D_H1})')
print(f'  After GCN layer 2    : {H2.shape}  (nodes × {D_H2})')
print(f'  Node 0 final emb     : {H2[0].round(4)}')

In [ ]:
# 1.2  GraphSAGE: Sample-and-Aggregate
class GraphSAGE:
    """
    GraphSAGE: sample fixed-size neighbourhood, aggregate (mean),
    concatenate with self embedding, apply linear transform.
    """
    def __init__(self, d_in, d_out, sample_size=3, seed=42):
        rng_gs = np.random.default_rng(seed)
        self.W_self   = rng_gs.normal(0, 0.1, (d_in,  d_out))
        self.W_neigh  = rng_gs.normal(0, 0.1, (d_in,  d_out))
        self.sample_k = sample_size

    def aggregate(self, H, A):
        """
        For each node: sample up to sample_k neighbours,
        compute mean of their features, then concat with self.
        """
        N        = len(H)
        H_out    = np.zeros((N, self.W_self.shape[1]))
        for v in range(N):
            neighbours = np.where(A[v] > 0)[0]
            if len(neighbours) > self.sample_k:
                neighbours = np.random.choice(neighbours, self.sample_k, replace=False)
            if len(neighbours) == 0:
                h_neigh = np.zeros_like(H[v])
            else:
                h_neigh = H[neighbours].mean(axis=0)
            h_self  = H[v]
            h_agg   = np.tanh(h_self @ self.W_self + h_neigh @ self.W_neigh)
            # L2 normalise
            norm    = np.linalg.norm(h_agg) + 1e-9
            H_out[v] = h_agg / norm
        return H_out

sage = GraphSAGE(d_in=D_IN, d_out=D_H1, sample_size=3)
H_sage = sage.aggregate(H0, A)
print(f'GraphSAGE output: {H_sage.shape}')
print(f'  L2 norms (should be ~1): {np.linalg.norm(H_sage, axis=1).round(4)}')

In [ ]:
# 1.3  Node Classification with GNN Features
# Use Karate Club as real graph
G_kc   = nx.karate_club_graph()
nodes  = list(G_kc.nodes())
N_kc   = len(nodes)

# Build adjacency matrix
A_kc   = np.array(nx.to_numpy_array(G_kc))

# Node features: degree + centrality measures
feat_degree  = np.array([G_kc.degree(n)              for n in nodes], dtype=float)
feat_between = np.array([nx.betweenness_centrality(G_kc)[n] for n in nodes], dtype=float)
feat_cluster = np.array([nx.clustering(G_kc)[n]       for n in nodes], dtype=float)
feat_page    = np.array([nx.pagerank(G_kc)[n]          for n in nodes], dtype=float)

H_kc  = np.column_stack([feat_degree, feat_between, feat_cluster, feat_page])
# Normalise features
H_kc  = (H_kc - H_kc.mean(0)) / (H_kc.std(0) + 1e-9)

# True labels
Y_kc  = np.array([1 if G_kc.nodes[n]['club']=='Mr. Hi' else 0 for n in nodes])

# Two GCN layers
rng_kc = np.random.default_rng(0)
W_kc1  = rng_kc.normal(0, 0.1, (4, 8))
W_kc2  = rng_kc.normal(0, 0.1, (8, 4))
H_kc1  = gcn_layer(H_kc,  A_kc, W_kc1)
H_kc2  = gcn_layer(H_kc1, A_kc, W_kc2)

# Linear classifier on GNN embeddings (LOO-CV due to small dataset)
from sklearn.model_selection import LeaveOneOut
loo   = LeaveOneOut()
preds = []
for tr, te in loo.split(H_kc2):
    lr_kc = LogisticRegression(max_iter=300)
    lr_kc.fit(H_kc2[tr], Y_kc[tr])
    preds.append(lr_kc.predict(H_kc2[te])[0])

acc_gnn = accuracy_score(Y_kc, preds)
print(f'Karate Club LOO accuracy (GNN features): {acc_gnn:.4f}  n={N_kc}')

In [ ]:
# 1.4  Link Prediction: Score Node Pairs
def link_score_dot(H, u, v):
    """Dot product of node embeddings as link probability."""
    return float(H[u] @ H[v])

def link_score_hadamard(H, u, v):
    """Element-wise product, then sum."""
    return float((H[u] * H[v]).sum())

# Existing edges (positive samples)
existing_edges = list(G_kc.edges())[:20]
# Non-edges (negative samples)
non_edges = list(nx.non_edges(G_kc))[:20]

scores_pos_dot = [link_score_dot(H_kc2, u, v) for u, v in existing_edges]
scores_neg_dot = [link_score_dot(H_kc2, u, v) for u, v in non_edges]

print(f'Link prediction (dot product scores):')
print(f'  Existing edges  mean: {np.mean(scores_pos_dot):.4f}')
print(f'  Non-edges       mean: {np.mean(scores_neg_dot):.4f}')
print(f'  Separation: {np.mean(scores_pos_dot) - np.mean(scores_neg_dot):.4f}  (positive = can discriminate)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# GNN embedding visualisation
from sklearn.decomposition import PCA
pca2 = PCA(n_components=2, random_state=42)
H_2d = pca2.fit_transform(H_kc2)
colors_kc = ['#534AB7' if l==1 else '#D85A30' for l in Y_kc]
nx_pos = nx.spring_layout(G_kc, seed=42)
nx.draw_networkx(G_kc, nx_pos, node_color=colors_kc, node_size=150,
                 font_size=7, ax=axes[0], edge_color='#D3D1C7')
axes[0].set_title('Karate Club Graph\n(blue=Mr.Hi, red=Officer)')
axes[0].axis('off')

axes[1].scatter(H_2d[Y_kc==0,0], H_2d[Y_kc==0,1], color='#D85A30', s=60, label='Officer', alpha=0.8)
axes[1].scatter(H_2d[Y_kc==1,0], H_2d[Y_kc==1,1], color='#534AB7', s=60, label='Mr. Hi', alpha=0.8)
axes[1].set_title('GNN Node Embeddings (PCA 2D)')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## Session 2 — Prototypical Networks (Few-Shot Learning)
**Time:** 10:30–11:00 AM | **Run time: ~5 sec**

In [ ]:
# 2.1  Prototypical Network
class PrototypicalNetwork:
    """
    N-way K-shot prototypical network:
    1. Compute prototype for each class = mean of K support embeddings
    2. Classify query as nearest prototype (Euclidean distance)
    3. Softmax over negative distances for probability estimates
    """
    def __init__(self, encoder=None):
        self.encoder = encoder  # optional feature extractor

    def compute_prototypes(self, support_X, support_y):
        """Returns dict: class → mean embedding."""
        classes    = np.unique(support_y)
        prototypes = {}
        for cls in classes:
            mask = support_y == cls
            emb  = support_X[mask]
            if self.encoder:
                emb = self.encoder(emb)
            prototypes[cls] = emb.mean(axis=0)
        return prototypes

    def predict(self, query_X, prototypes):
        classes = sorted(prototypes.keys())
        P       = np.array([prototypes[c] for c in classes])
        if self.encoder:
            query_X = self.encoder(query_X)
        # Euclidean distances to each prototype
        dists   = np.array([
            [np.sum((q - p)**2) for p in P]
            for q in query_X
        ])  # (n_query, n_classes)
        # Softmax over negative distances
        logits  = -dists
        logits -= logits.max(axis=1, keepdims=True)
        probs   = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
        pred_idx = dists.argmin(axis=1)
        preds    = np.array([classes[i] for i in pred_idx])
        return preds, probs

# 5-way 5-shot episode
rng_p = np.random.default_rng(42)
N_WAY_P, K_SHOT_P, D_P, N_QUERY_P = 5, 5, 32, 10

# Each class has a distinct cluster centre
centres = rng_p.normal(0, 3, (N_WAY_P, D_P))
X_support = np.vstack([
    centres[c] + rng_p.normal(0, 0.5, (K_SHOT_P, D_P))
    for c in range(N_WAY_P)
])
y_support = np.repeat(np.arange(N_WAY_P), K_SHOT_P)

X_query   = np.vstack([
    centres[c] + rng_p.normal(0, 0.5, (N_QUERY_P, D_P))
    for c in range(N_WAY_P)
])
y_query   = np.repeat(np.arange(N_WAY_P), N_QUERY_P)

pnet = PrototypicalNetwork()
proto = pnet.compute_prototypes(X_support, y_support)
preds_p, probs_p = pnet.predict(X_query, proto)
acc_p = accuracy_score(y_query, preds_p)
print(f'{N_WAY_P}-way {K_SHOT_P}-shot Prototypical Network:')
print(f'  Accuracy: {acc_p:.4f}')
print(f'  Mean confidence: {probs_p.max(axis=1).mean():.4f}')

In [ ]:
# 2.2  Episode Training Loop
def sample_episode(X_all, y_all, n_way=3, k_shot=5, n_query=10, rng_ep=None):
    """Sample a random N-way K-shot episode."""
    if rng_ep is None:
        rng_ep = np.random.default_rng()
    classes    = rng_ep.choice(np.unique(y_all), n_way, replace=False)
    sup_X, sup_y, qry_X, qry_y = [], [], [], []
    for i, cls in enumerate(classes):
        idx = np.where(y_all == cls)[0]
        chosen = rng_ep.choice(idx, k_shot + n_query, replace=False)
        sup_idx = chosen[:k_shot]
        qry_idx = chosen[k_shot:]
        sup_X.append(X_all[sup_idx]); sup_y.extend([i]*k_shot)
        qry_X.append(X_all[qry_idx]); qry_y.extend([i]*n_query)
    return (np.vstack(sup_X), np.array(sup_y),
            np.vstack(qry_X), np.array(qry_y))

# Run 50 episodes on Wine dataset
X_wine, y_wine = load_wine(return_X_y=True)
X_wine_sc      = StandardScaler().fit_transform(X_wine)

epoch_accs = []
rng_ep     = np.random.default_rng(42)
for ep in range(50):
    sX, sy, qX, qy = sample_episode(X_wine_sc, y_wine,
                                      n_way=3, k_shot=5, n_query=8, rng_ep=rng_ep)
    proto_ep = pnet.compute_prototypes(sX, sy)
    preds_ep, _ = pnet.predict(qX, proto_ep)
    epoch_accs.append(accuracy_score(qy, preds_ep))

print(f'Prototypical Network — 50 episodes on Wine (3-way 5-shot):')
print(f'  Mean accuracy: {np.mean(epoch_accs):.4f}')
print(f'  Std accuracy : {np.std(epoch_accs):.4f}')
print(f'  Best episode : {max(epoch_accs):.4f}')
print(f'  Worst episode: {min(epoch_accs):.4f}')

---
## Session 3 — MAML + Continual Learning
**Time:** 11:00 AM–1:00 PM | **Run time: ~18 sec**

In [ ]:
# 3.1  MAML: Model-Agnostic Meta-Learning (Linear Regression Simulation)
rng = np.random.default_rng(42)
D_MAML = 6

def task_loss(w, X, y):
    pred = X @ w
    return np.mean((pred - y)**2)

def task_grad(w, X, y):
    pred = X @ w
    return 2 * X.T @ (pred - y) / len(y)

def inner_update(w, X, y, lr=0.05, n_steps=3):
    """Fast adaptation: K gradient steps on support set."""
    w_adapted = w.copy()
    for _ in range(n_steps):
        g         = task_grad(w_adapted, X, y)
        w_adapted = w_adapted - lr * g
    return w_adapted

def generate_task(rng_t, n=20, d=6):
    """Each task has a different true weight vector."""
    w_true = rng_t.normal(0, 1, d)
    X      = rng_t.normal(0, 1, (n, d))
    y      = X @ w_true + rng_t.normal(0, 0.1, n)
    return X, y, w_true

# Meta-training
w_meta      = rng.normal(0, 0.1, D_MAML)  # meta-initialisation
META_LR     = 0.005
N_META_STEPS = 200
N_TASKS_PER_STEP = 4

maml_train_losses = []
maml_test_losses  = []

for meta_step in range(N_META_STEPS):
    meta_grad = np.zeros_like(w_meta)
    step_train_loss = 0.0
    step_test_loss  = 0.0

    for _ in range(N_TASKS_PER_STEP):
        X_task, y_task, _ = generate_task(rng)
        X_support, X_query = X_task[:10], X_task[10:]
        y_support, y_query = y_task[:10], y_task[10:]

        # Inner loop: fast adapt on support set
        w_adapted = inner_update(w_meta, X_support, y_support)

        # Outer loop: compute loss on query set with adapted weights
        g_query = task_grad(w_adapted, X_query, y_query)
        meta_grad     += g_query
        step_train_loss += task_loss(w_meta,     X_support, y_support)
        step_test_loss  += task_loss(w_adapted,  X_query,   y_query)

    # Meta-update
    w_meta        -= META_LR * meta_grad / N_TASKS_PER_STEP
    maml_train_losses.append(step_train_loss / N_TASKS_PER_STEP)
    maml_test_losses.append(step_test_loss  / N_TASKS_PER_STEP)

print(f'MAML Meta-Training ({N_META_STEPS} steps):')
print(f'  Initial train loss: {maml_train_losses[0]:.4f}  → Final: {maml_train_losses[-1]:.4f}')
print(f'  Initial  test loss: {maml_test_losses[0]:.4f}  → Final: {maml_test_losses[-1]:.4f}')

In [ ]:
# 3.2  EWC: Elastic Weight Consolidation
class EWCModel:
    """
    EWC: after training on task A, compute Fisher information
    to measure parameter importance. When training on task B,
    add penalty: λ * Σ F_i (θ_i - θ_A_i)^2
    """
    def __init__(self, n_features, lr=0.01):
        self.w    = np.zeros(n_features + 1)  # weights + bias
        self.lr   = lr
        self.theta_star = None   # weights after task A
        self.fisher     = None   # Fisher information
        self.ewc_lambda = 100.0

    def forward(self, X):
        X_bias = np.column_stack([X, np.ones(len(X))])
        return 1 / (1 + np.exp(-np.clip(X_bias @ self.w, -30, 30)))

    def compute_loss(self, X, y, ewc=False):
        prob  = self.forward(X)
        ce    = -np.mean(y * np.log(prob + 1e-9) + (1-y) * np.log(1-prob + 1e-9))
        if ewc and self.theta_star is not None:
            penalty = 0.5 * self.ewc_lambda * np.sum(
                self.fisher * (self.w - self.theta_star)**2
            )
            return ce + penalty
        return ce

    def compute_grad(self, X, y, ewc=False):
        X_bias = np.column_stack([X, np.ones(len(X))])
        prob   = self.forward(X)
        grad   = X_bias.T @ (prob - y) / len(y)
        if ewc and self.theta_star is not None:
            grad += self.ewc_lambda * self.fisher * (self.w - self.theta_star)
        return grad

    def fit_task(self, X, y, n_epochs=50, ewc=False):
        for _ in range(n_epochs):
            g       = self.compute_grad(X, y, ewc=ewc)
            self.w -= self.lr * g

    def consolidate(self, X, y, n_samples=100):
        """Compute diagonal Fisher information after training on task A."""
        self.theta_star = self.w.copy()
        X_bias = np.column_stack([X, np.ones(len(X))])
        fisher = np.zeros_like(self.w)
        for i in range(min(n_samples, len(X))):
            prob = self.forward(X[[i]])
            g    = X_bias[i] * (prob[0] - y[i])
            fisher += g**2
        self.fisher = fisher / min(n_samples, len(X))

# Generate two tasks
rng_ewc = np.random.default_rng(42)
X_A, y_A = make_classification(300, n_features=10, random_state=0)
X_B, y_B = make_classification(300, n_features=10, random_state=1)  # different task

X_A_tr, X_A_te, y_A_tr, y_A_te = train_test_split(X_A, y_A, test_size=0.3, random_state=42)
X_B_tr, X_B_te, y_B_tr, y_B_te = train_test_split(X_B, y_B, test_size=0.3, random_state=42)

# Without EWC
model_no_ewc = EWCModel(n_features=10)
model_no_ewc.fit_task(X_A_tr, y_A_tr, n_epochs=100)
acc_A_before = accuracy_score(y_A_te, (model_no_ewc.forward(X_A_te) > 0.5).astype(int))
model_no_ewc.fit_task(X_B_tr, y_B_tr, n_epochs=100)  # overwrite
acc_A_after  = accuracy_score(y_A_te, (model_no_ewc.forward(X_A_te) > 0.5).astype(int))

# With EWC
model_ewc = EWCModel(n_features=10)
model_ewc.fit_task(X_A_tr, y_A_tr, n_epochs=100)
model_ewc.consolidate(X_A_tr, y_A_tr)
acc_A_ewc_before = accuracy_score(y_A_te, (model_ewc.forward(X_A_te) > 0.5).astype(int))
model_ewc.fit_task(X_B_tr, y_B_tr, n_epochs=100, ewc=True)
acc_A_ewc_after  = accuracy_score(y_A_te, (model_ewc.forward(X_A_te) > 0.5).astype(int))

print('EWC — Preventing Catastrophic Forgetting:')
print(f'  Without EWC: Task A acc {acc_A_before:.4f} → {acc_A_after:.4f}  '
      f'(forgetting: {acc_A_before-acc_A_after:.4f})')
print(f'  With    EWC: Task A acc {acc_A_ewc_before:.4f} → {acc_A_ewc_after:.4f}  '
      f'(forgetting: {acc_A_ewc_before-acc_A_ewc_after:.4f})')

In [ ]:
# 3.3  Experience Replay Buffer
class ReplayBuffer:
    """
    Store a reservoir of past samples.
    When training on new task, interleave with replayed samples.
    Uses reservoir sampling to maintain a uniform random sample.
    """
    def __init__(self, max_size=200):
        self.max_size = max_size
        self.X_buf    = []
        self.y_buf    = []
        self.n_seen   = 0

    def add(self, X, y):
        """Reservoir sampling: each new sample has max_size/n_seen chance of staying."""
        for xi, yi in zip(X, y):
            self.n_seen += 1
            if len(self.X_buf) < self.max_size:
                self.X_buf.append(xi)
                self.y_buf.append(yi)
            else:
                j = np.random.randint(self.n_seen)
                if j < self.max_size:
                    self.X_buf[j] = xi
                    self.y_buf[j] = yi

    def sample(self, n):
        """Sample n examples from buffer."""
        n      = min(n, len(self.X_buf))
        idx    = np.random.choice(len(self.X_buf), n, replace=False)
        X_samp = np.array([self.X_buf[i] for i in idx])
        y_samp = np.array([self.y_buf[i] for i in idx])
        return X_samp, y_samp

    def __len__(self):
        return len(self.X_buf)

# Continual learning with replay
buffer    = ReplayBuffer(max_size=150)
buffer.add(X_A_tr, y_A_tr)   # store task A samples

# Train on task B while replaying task A
model_replay = EWCModel(n_features=10)
model_replay.fit_task(X_A_tr, y_A_tr, n_epochs=80)
acc_replay_before = accuracy_score(y_A_te, (model_replay.forward(X_A_te) > 0.5).astype(int))

# Interleaved training on B + replayed A
for epoch in range(100):
    X_replay, y_replay = buffer.sample(30)
    X_mixed  = np.vstack([X_B_tr[:60], X_replay])
    y_mixed  = np.concatenate([y_B_tr[:60], y_replay])
    # Shuffle
    idx_m    = np.random.permutation(len(X_mixed))
    g        = model_replay.compute_grad(X_mixed[idx_m], y_mixed[idx_m])
    model_replay.w -= model_replay.lr * g

acc_replay_after = accuracy_score(y_A_te, (model_replay.forward(X_A_te) > 0.5).astype(int))
print(f'  Replay buffer: Task A acc {acc_replay_before:.4f} → {acc_replay_after:.4f}  '
      f'(forgetting: {acc_replay_before-acc_replay_after:.4f})')

---
## Lunch Break — 1:00–2:00 PM
1. Why does GCN normalise the adjacency matrix with D^(-1/2) from both sides?
2. What makes prototypical networks more efficient than standard k-NN for few-shot?
3. How does Fisher information determine which weights matter most in EWC?
4. When is replay buffer preferable to EWC for continual learning?
---

## Session 5 — Research Capstone: GNN + Meta-Learning + Continual System
**Time:** 2:00–4:00 PM | **Run time: ~20 sec**

In [ ]:
# 5.1  GNN as Feature Extractor for Few-Shot Classification
print('='*60)
print(' RESEARCH CAPSTONE: GNN + META + CONTINUAL SYSTEM')
print('='*60)

# Build a larger synthetic graph for experiments
rng_cap = np.random.default_rng(42)
N_CAP   = 50
D_NODE  = 8

# Create random graph with 3 communities
G_cap = nx.planted_partition_graph(l=3, k=17, p_in=0.6, p_out=0.1, seed=42)
nodes_cap = list(G_cap.nodes())
A_cap     = np.array(nx.to_numpy_array(G_cap))

# Node features: community-dependent
y_cap = np.array([G_cap.nodes[n].get('block', 0) for n in nodes_cap])
H_cap_raw = rng_cap.normal(0, 1, (len(nodes_cap), D_NODE))
# Add community signal
for cls in range(3):
    mask = y_cap == cls
    H_cap_raw[mask] += cls * 1.5

# Apply 2-layer GCN
W_cap1 = rng_cap.normal(0, 0.1, (D_NODE, 12))
W_cap2 = rng_cap.normal(0, 0.1, (12, 8))
H_cap1 = gcn_layer(H_cap_raw, A_cap, W_cap1)
H_cap2 = gcn_layer(H_cap1,    A_cap, W_cap2)

print(f'Graph: {len(nodes_cap)} nodes, {G_cap.number_of_edges()} edges, {len(np.unique(y_cap))} communities')
print(f'GNN output: {H_cap2.shape}')

In [ ]:
# 5.2  Few-Shot Node Classification with GNN Features
np.random.seed(42)

# Compare: raw features vs GNN features for few-shot
for k_shot_test in [1, 3, 5, 10]:
    accs_raw, accs_gnn = [], []
    for trial in range(30):
        # Sample support and query
        support_X_raw, support_X_gnn, support_y = [], [], []
        query_X_raw,   query_X_gnn,   query_y   = [], [], []
        for cls in range(3):
            idx = np.where(y_cap == cls)[0]
            chosen = np.random.choice(idx, k_shot_test + 5, replace=False)
            support_X_raw.append(H_cap_raw[chosen[:k_shot_test]])
            support_X_gnn.append(H_cap2[chosen[:k_shot_test]])
            support_y.extend([cls]*k_shot_test)
            query_X_raw.append(H_cap_raw[chosen[k_shot_test:]])
            query_X_gnn.append(H_cap2[chosen[k_shot_test:]])
            query_y.extend([cls]*5)

        sX_raw  = np.vstack(support_X_raw); sy = np.array(support_y)
        qX_raw  = np.vstack(query_X_raw);   qy = np.array(query_y)
        sX_gnn  = np.vstack(support_X_gnn)
        qX_gnn  = np.vstack(query_X_gnn)

        proto_raw = pnet.compute_prototypes(sX_raw, sy)
        proto_gnn = pnet.compute_prototypes(sX_gnn, sy)
        p_raw, _ = pnet.predict(qX_raw, proto_raw)
        p_gnn, _ = pnet.predict(qX_gnn, proto_gnn)
        accs_raw.append(accuracy_score(qy, p_raw))
        accs_gnn.append(accuracy_score(qy, p_gnn))

    print(f'k={k_shot_test:2d}: Raw={np.mean(accs_raw):.4f}  GNN={np.mean(accs_gnn):.4f}  '
          f'Gain={np.mean(accs_gnn)-np.mean(accs_raw):+.4f}')

In [ ]:
# 5.3  Continual Node Classification: Two Graph Tasks
# Task A: classify nodes in graph_cap communities
# Task B: classify nodes on a new graph (different structure)

G_new  = nx.planted_partition_graph(l=3, k=15, p_in=0.55, p_out=0.12, seed=99)
nodes_new = list(G_new.nodes())
A_new     = np.array(nx.to_numpy_array(G_new))
y_new     = np.array([G_new.nodes[n].get('block', 0) for n in nodes_new])

H_new_raw = rng_cap.normal(0, 1, (len(nodes_new), D_NODE))
for cls in range(3):
    H_new_raw[y_new==cls] += cls * 1.2

W_new1 = rng_cap.normal(0, 0.1, (D_NODE, 12))
W_new2 = rng_cap.normal(0, 0.1, (12, 8))
H_new1 = gcn_layer(H_new_raw, A_new, W_new1)
H_new2 = gcn_layer(H_new1,    A_new, W_new2)

# Train logistic classifier on GNN features for both tasks
X_A_gnn_tr, X_A_gnn_te, yA_tr, yA_te = train_test_split(H_cap2, y_cap, test_size=0.3, random_state=42)
X_B_gnn_tr, X_B_gnn_te, yB_tr, yB_te = train_test_split(H_new2, y_new, test_size=0.3, random_state=42)

# No continual learning
m_seq = LogisticRegression(max_iter=300).fit(X_A_gnn_tr, yA_tr)
acc_A_seq_before = m_seq.score(X_A_gnn_te, yA_te)
m_seq.fit(X_B_gnn_tr, yB_tr)  # overwrites
acc_A_seq_after  = m_seq.score(X_A_gnn_te, yA_te)

# With replay: mix task A and B training data
buf_cl = ReplayBuffer(max_size=100)
buf_cl.add(X_A_gnn_tr, yA_tr)
X_rpl, y_rpl = buf_cl.sample(50)
X_mix = np.vstack([X_B_gnn_tr, X_rpl])
y_mix = np.concatenate([yB_tr, y_rpl])
m_replay_cl = LogisticRegression(max_iter=300).fit(X_mix, y_mix)
acc_A_replay = m_replay_cl.score(X_A_gnn_te, yA_te)

print('Continual Node Classification:')
print(f'  Sequential (no CL): Task A {acc_A_seq_before:.4f} → {acc_A_seq_after:.4f}  '
      f'forgetting={acc_A_seq_before-acc_A_seq_after:.4f}')
print(f'  Replay buffer     : Task A retained at {acc_A_replay:.4f}  '
      f'forgetting={acc_A_seq_before-acc_A_replay:.4f}')

In [ ]:
# Final Visualisation
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. MAML convergence
axes[0,0].plot(maml_train_losses, color='#534AB7', linewidth=1.8, label='Train loss')
axes[0,0].plot(maml_test_losses,  color='#1D9E75', linewidth=1.8, label='Query loss')
axes[0,0].set_title('MAML Meta-Training Convergence')
axes[0,0].set_xlabel('Meta-step'); axes[0,0].set_ylabel('MSE')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# 2. Prototypical network episode accs
axes[0,1].hist(epoch_accs, bins=15, color='#D85A30', alpha=0.85, edgecolor='white')
axes[0,1].axvline(np.mean(epoch_accs), color='black', linestyle='--', linewidth=2,
                   label=f'Mean={np.mean(epoch_accs):.3f}')
axes[0,1].set_title('Prototypical Network: Episode Accuracies')
axes[0,1].set_xlabel('Episode accuracy'); axes[0,1].legend()

# 3. GNN embedding space
from sklearn.decomposition import PCA
pca_cap = PCA(n_components=2, random_state=42)
H_cap_2d = pca_cap.fit_transform(H_cap2)
for cls, color in [(0,'#534AB7'),(1,'#1D9E75'),(2,'#D85A30')]:
    m = y_cap == cls
    axes[1,0].scatter(H_cap_2d[m,0], H_cap_2d[m,1], color=color,
                      s=50, alpha=0.8, label=f'Community {cls}')
axes[1,0].set_title('GNN Node Embeddings (3 communities)')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# 4. Forgetting comparison
cats     = ['Sequential\n(no CL)', 'EWC', 'Replay Buffer']
forgetting = [
    acc_A_seq_before  - acc_A_seq_after,
    acc_A_ewc_before  - acc_A_ewc_after,
    acc_A_seq_before  - acc_A_replay,
]
colors_f = ['#D85A30', '#534AB7', '#1D9E75']
bars = axes[1,1].bar(cats, forgetting, color=colors_f, alpha=0.85)
axes[1,1].bar_label(bars, fmt='%.4f', padding=3)
axes[1,1].set_title('Catastrophic Forgetting Comparison')
axes[1,1].set_ylabel('Accuracy drop on Task A')
axes[1,1].axhline(0, color='black', linewidth=0.8)

plt.suptitle('Day 21 Capstone: GNN + Meta-Learning + Continual Learning', fontsize=12)
plt.tight_layout(); plt.show()

---
## Day 21 Summary

| Concept | Skill Gained |
|---|---|
| GCN layer | Symmetric normalised adjacency, message aggregation |
| GraphSAGE | Fixed-size neighbourhood sampling, L2-normalised output |
| Node classification | LOO-CV on GNN embeddings, Karate Club |
| Link prediction | Dot product and Hadamard edge scoring |
| Prototypical network | N-way K-shot, prototype mean, Euclidean distance |
| Episode training | Random episode sampling, 50-episode accuracy report |
| MAML | Inner loop fast adapt, outer loop meta-gradient update |
| EWC | Fisher information, weight consolidation, forgetting measurement |
| Replay buffer | Reservoir sampling, interleaved training on old + new tasks |
| Continual node class | Sequential vs EWC vs Replay on two graph tasks |

---
## Day 22 Preview
- Sparse autoencoders: interpretable feature dictionaries
- Mechanistic interpretability concepts
- Uncertainty estimation: deep ensembles and MC Dropout
- Selective prediction: abstain when uncertain
- Final week capstone research project

In [ ]:
# Bonus: Day 21 in one cell
rng_b = np.random.default_rng(0)
N_b, D_b = 10, 6
A_b = (rng_b.random((N_b,N_b)) > 0.7).astype(float)
A_b = np.triu(A_b,1); A_b += A_b.T   # symmetric
H_b = rng_b.normal(0,1,(N_b,D_b))
W_b = rng_b.normal(0,0.1,(D_b,4))
H_b2 = gcn_layer(H_b, A_b, W_b)
print(f'GCN layer output: {H_b2.shape}')

# Prototypical network quick test
rng_pb = np.random.default_rng(42)
N_WAY_B, K_SHOT_B, D_PB = 3, 5, 16
centres_b = rng_pb.normal(0, 3, (N_WAY_B, D_PB))
sp_b = np.vstack([centres_b[c]+rng_pb.normal(0,.5,(K_SHOT_B,D_PB)) for c in range(N_WAY_B)])
sy_b = np.repeat(np.arange(N_WAY_B), K_SHOT_B)
qr_b = np.vstack([centres_b[c]+rng_pb.normal(0,.5,(5,D_PB)) for c in range(N_WAY_B)])
qy_b = np.repeat(np.arange(N_WAY_B), 5)
pt_b = pnet.compute_prototypes(sp_b, sy_b)
pr_b,_ = pnet.predict(qr_b, pt_b)
print(f'{N_WAY_B}-way {K_SHOT_B}-shot accuracy: {accuracy_score(qy_b,pr_b):.4f}')
print(f'MAML final meta-loss: {maml_test_losses[-1]:.4f}')
print(f'EWC forgetting: {acc_A_ewc_before-acc_A_ewc_after:.4f}  Replay: {acc_A_seq_before-acc_A_replay:.4f}')
print('\nDay 21 complete — GNNs, meta-learning, continual learning!')